[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/estimation/kalman/kalman.ipynb)

# Tracking a Pose with a Kalman Filter

This example tracks a vehicle's pose in the plane with a Kalman filter. The vehicle's odometry, the commands it gave, drifts further from the truth with every step; a few times along the way it gets a noisy reading of its whole pose, position and heading. The filter blends the two: it carries its best guess of the pose forward with the commands, keeps track of how uncertain that guess has become, and at each reading moves toward it by as much as the two uncertainties warrant.

The pose is a motor, and an error in it is a small twist on its right. The uncertainty is a covariance of those twists, typed as a map from a line, a linear readout of a twist through their pairing, to the twist correlated with it, `Twist <- Line`. Being a map, it moves through each step of the drive the way any map does, its readout pulled through the step and its twist pushed back, `step << sigma(step >> Line)`, with no Jacobian to derive. The gain at a reading is a ratio of two such maps, and the position ellipse is the covariance read through the map from twists to the motion of the vehicle's centre.

In the matrix notation of the extended Kalman filter on $\mathrm{SE}(2)$ the covariance reads as a $3 \times 3$ matrix $P$, propagated as $F P F^{\mathsf T} + Q$ with the adjoint matrix $F$ of each step and updated with the Kalman gain $K = P (P + R)^{-1}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA2D as ga
from examples.util.animation import show_animation
from examples.estimation.kalman import render

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.vector()                          # Vector+
# A linear readout of a twist is an antibivector line; a covariance maps it to the correlated twist.
Line = ga.gatype.antibivector()                     # Antibivector-
# A pose perturbation, the twist on the right that carries the estimate to the truth.
Twist = ga.gatype.bivector()                        # Bivector+
Motor = ga.gatype.rotor()                           # Rotor+
Covariance = ga.gatype((Twist, Line))
Quadric = ga.gatype((Plane, Point))

## 1. Predicting and correcting

The filter alternates between predicting and correcting. A prediction carries the estimate through a commanded step and moves the covariance with it, the way any map moves, `step << sigma(step >> Line)`, adding the motion noise. A correction at a reading takes the innovation, the twist from the estimate to the reading; the gain, a ratio of two covariance maps, takes it to the twist to apply, and the covariance shrinks by what the reading told. Both noises are covariances built from the twists of translation and of rotation about the body's origin.

In [ ]:
def covariance(translation_std: float, rotation_std: float) -> Covariance:
    """A body-frame noise covariance is isotropic in translation and in rotation about the body's origin."""
    translation = mv.yw * (mv.yw & Line) + mv.wx * (mv.wx & Line)               # [] Twist+ <- Line-
    rotation = mv.xy * (mv.xy & Line)                                           # [] Twist+ <- Line-
    return translation * translation_std**2 + rotation * rotation_std**2


def update(estimate: Motor, sigma: Covariance, reading: Motor, measurement_noise: Covariance) -> tuple[Motor, Covariance]:
    """A noisy pose reading, its error a right perturbation with covariance `measurement_noise`, corrects the estimate and shrinks its covariance."""
    # The innovation is the log of the relative motor, the gain is a ratio of covariance maps, and the
    # correction is exponentiated.
    innovation = (estimate.inverse() * reading).log() * 2                       # [] Twist+
    gain = sigma((sigma + measurement_noise).inverse())                         # [] Twist+ <- Twist+
    correction = (gain(innovation) * 0.5).exp()                                 # [] Motor+
    return estimate * correction, sigma - gain(sigma)

## 2. Simulating the drive

The vehicle drives at unit speed along its forward axis while its turn rate wanders as a sine wave. The true motion adds a draw of the motion noise to every commanded step, dead reckoning follows the commands alone, and the filter predicts through every step and corrects after each group of them, at a reading. Drawing twists from a covariance needs readouts that are independent under it: lines orthonormal in the covariance's own form, each carrying an independent normal draw.

In [ ]:
# The time step, the readings and the commanded steps between them, and the seed of the noise.
dt, n_readings, n_steps_per_reading, seed = 0.1, 12, 25, 2
# The commanded turn rate's amplitude and angular frequency.
turn_amplitude, turn_frequency = 0.9, 0.2
# Standard deviations of translation and rotation, of the motion per square root of time and of a reading.
motion_std = np.array([0.05, 0.12])
measurement_std = np.array([0.3, 0.1])

In [ ]:
def sample(noise: Covariance, normals: np.ndarray) -> Twist:
    """The sampled twists have covariance `noise`: readouts orthonormal in its own form are independent under it,
    so the twists they select, `noise(lines)`, weighted by the independent standard normal draws `[..., 3]`,
    reproduce it, `(noise(lines) * (noise(lines) & Line)).sum(axis=0) == noise`.
    """
    readouts = Line & noise                                                     # [] Scalar+ <- (Line-, Line-)
    # Lines orthonormal in the form itself.
    _, lines = readouts.eigh(readouts)                                          # [modes] Line-
    return (noise(lines) * normals).sum(axis=-1)                                # [...] Twist+


def drive(initial: Motor, increments: Twist, motion_noise: Covariance, measurement_noise: Covariance,
          rng: np.random.Generator) -> Iterator[tuple[Motor, Motor, Motor, Covariance, Motor]]:
    """The drive yields, at every commanded step and after every reading, the true pose, the dead-reckoned pose,
    the filter's estimate and covariance, and the latest reading."""
    truth = dead = estimate = reading = initial
    sigma = covariance(0.0, 0.0)
    for segment in increments:
        for increment in segment:
            step = (increment * 0.5).exp()                                          # [] Motor+
            noisy_step = ((increment + sample(motion_noise, rng.normal(size=3))) * 0.5).exp()  # [] Motor+
            truth = truth * noisy_step
            dead = dead * step
            # Carry the readout through the step and push the correlated twist back.
            estimate, sigma = estimate * step, (step << sigma(step >> Line)) + motion_noise
            yield truth, dead, estimate, sigma, reading
        measurement_error = (sample(measurement_noise, rng.normal(size=3)) * 0.5).exp()  # [] Motor+
        reading = truth * measurement_error
        estimate, sigma = update(estimate, sigma, reading, measurement_noise)
        yield truth, dead, estimate, sigma, reading


times = np.arange(n_readings * n_steps_per_reading) * dt
turn = turn_amplitude * np.sin(turn_frequency * times)
increments = ((mv.xy * turn - mv.wx) * dt).reshape(n_readings, n_steps_per_reading)  # [readings, steps] Twist+
motion_noise = covariance(*(motion_std * np.sqrt(dt)))                          # [] Twist+ <- Line-
measurement_noise = covariance(*measurement_std)                                # [] Twist+ <- Line-
states = drive(mv.rotor(), increments, motion_noise, measurement_noise, np.random.default_rng(seed))
truth, dead, estimates, covariances, readings = (stack(path) for path in zip(*states))  # [states] Motor+, Motor+, Motor+, Twist+ <- Line-, Motor+

In [ ]:
print("the motion noise per step, a map from lines to twists:\n", motion_noise.coefficient_table())
print("the covariance after the last reading:\n", covariances[-1].coefficient_table())

## 3. Position uncertainty and tracking animation

The covariance is one of twists, but what the eye reads is where the vehicle might be. The map from twists to the motion of the vehicle's centre carries the covariance to the position, and its adjugate pulls the planes that read a position back to the lines that read a twist; the position's second moment then has an inverse that is the quadric of the confidence ellipse.

In the animation, the true path is drawn in black, the dead-reckoned path in dashed red, and the filtered path in blue, each with its vehicle outline; green crosses mark the pose readings received so far, and the shaded blue quadric shows the filter's two-standard-deviation position uncertainty growing between readings and shrinking at each update.

In [ ]:
def position_ellipse(estimate: Motor, sigma: Covariance, positions: Point, sigmas: float) -> Quadric:
    """The quadric of each estimated position's uncertainty vanishes on its confidence ellipse, `sigmas` standard deviations out."""
    # A twist moves the position by its commutator with it: a map from twists to displacements.
    shift = Twist.commutator(positions)(estimate >> Twist)                      # [states] Point- <- Twist+
    # Its adjugate pulls each plane back to the line that reads the same displacement from the twist.
    readout = shift.adjugate()                                                  # [states] Line- <- Plane+
    # The covariance pushed forward through the motion adds to the position's own dyad: its second moment.
    moment = positions * (Plane & positions) + shift(sigma(readout))            # [states] Point- <- Plane+
    return moment.inverse() - (1 + sigmas**2) * mv.w * (mv.w & Point)           # [states] Plane+ <- Point-


origin = mv.w.dual()                                                            # [] Point-
sigmas = 2.0
positions = estimates >> origin                                                # [states] Point-
ellipses = position_ellipse(estimates, covariances, positions, sigmas)         # [states] Plane+ <- Point-
# The distance between two points of unit weight is the norm of the line joining them.
true_positions = truth >> origin                                                # [states] Point-
dead_error = (true_positions & (dead >> origin)).norm()                         # [states] Scalar+
filtered_error = (true_positions & positions).norm()                            # [states] Scalar+
frame_duration_ms = 40
show_animation(render.animate_tracking(truth, dead, estimates, positions, ellipses, readings), "kalman", frame_duration_ms)

In [ ]:
print("mean position error, dead reckoning:", dead_error.mean(axis=0).to_array())
print("mean position error, filtered:      ", filtered_error.mean(axis=0).to_array())

In [ ]:
# checks
from examples.estimation.kalman import core

core_ellipses = core.position_ellipse(estimates, covariances, positions, sigmas)
np.testing.assert_allclose((ellipses - core_ellipses).kernel, 0.0, atol=1e-12)
# Sparse noisy readings keep the filtered path far closer to the truth than the dead-reckoned one, and the
# covariance stays its own adjugate through the filter.
assert filtered_error.mean(axis=0) < 0.25 * dead_error.mean(axis=0)
np.testing.assert_allclose(covariances.adjugate().kernel, covariances.kernel, atol=1e-12)